<a href="https://colab.research.google.com/github/prjhseo/movieLens_recommendation/blob/main/Model_based_Rating_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!wget https://files.grouplens.org/datasets/movielens/ml-25m.zip
!unzip ml-25m.zip

--2026-06-15 05:39:27--  https://files.grouplens.org/datasets/movielens/ml-25m.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 261978986 (250M) [application/zip]
Saving to: ‘ml-25m.zip’

ml-25m.zip          100%[===================>] 249.84M  48.1MB/s    in 5.6s    

2026-06-15 05:39:33 (44.3 MB/s) - ‘ml-25m.zip’ saved [261978986/261978986]

Archive:  ml-25m.zip
   creating: ml-25m/
  inflating: ml-25m/tags.csv         
  inflating: ml-25m/links.csv        
  inflating: ml-25m/README.txt       
  inflating: ml-25m/ratings.csv      
  inflating: ml-25m/genome-tags.csv  
  inflating: ml-25m/genome-scores.csv  
  inflating: ml-25m/movies.csv       


# 데이터 로드
ratings.csv에서 각 열을 각각 users,items,ratings에 numpy array 형태로 저장 (numpy function을 이용하기 위해 )

In [2]:
import numpy as np

users=[]
items=[]
ratings=[]

with open("ml-25m/ratings.csv","r") as f:
    print(f.readline()) # skip column names

    for line in f:
        uid,mid,rating,_=line.split(",")

        users.append(int(uid))
        items.append(int(mid))
        ratings.append(float(rating))

users=np.array(users)
items=np.array(items)
ratings=np.array(ratings)

userId,movieId,rating,timestamp



# 기본 모델의 RMSE 확인
- 기본 모델 : f(u,i)=α
    - 평균에 수렴
- RMSE : (실제값-예측값)^2 의 평균의 제곱근

In [3]:
mean=ratings.mean() # 예측값

rmse=((ratings-mean)**2).mean() **0.5
print(rmse)

1.0607439399275531


# Bias 모델
f(u,i)= α + βu + βi  
βu : 사용자 u의 bias (사용자 평균에 비해 얼마나 더 점수를 잘 주는지)  
βi : 아이템 i의 bias (아이템이 평균에 비해 점수 받는 정도)

## 변수 생성 및 초기화

In [4]:
alpha=ratings.mean()
user_bias=np.zeros(users.max()+1) # user의 index가 1부터 시작
item_bias=np.zeros(items.max()+1)

## Gradient Descent

In [5]:
lr=1 # learning rate
lmd=0.001 # lamda

n_ratings=len(ratings)
n_users=len(users)
n_items=len(items)

for epoch in range(100):

    h= alpha + user_bias[users]+item_bias[items] # 예측 평점
    diff = h- ratings

    rmse=(diff*2).mean()**0.5
    print(f"epoch: {epoch},rmse: {rmse}")

    # 미분값
    # 크기 커지는 것을 방지하기 위해 합을 평균으로 대체
    grd_alpha=diff.mean()
    grd_user_bias= np.bincount(users,weights=diff)/n_ratings + lmd*user_bias/n_users # 숫자 2 생략
    grad_item_bias= np.bincount(items,weights=diff)/n_ratings + lmd*item_bias/n_items

    alpha-=lr*grd_alpha
    user_bias-=lr*grd_user_bias
    item_bias-=lr*grad_item_bias

h=alpha+user_bias[users]+item_bias[items]
diff=h-ratings

rmse=(diff**2).mean()**0.5
print(f"rmse: {rmse},alpha: {alpha}")

epoch: 0,rmse: 8.27994263877069e-09
epoch: 1,rmse: 0.016416311859249737
epoch: 2,rmse: 0.016398174057814404
epoch: 3,rmse: 0.016385173408865396
epoch: 4,rmse: 0.01637218704341167
epoch: 5,rmse: 0.01635921812532143
epoch: 6,rmse: 0.016346266625770545
epoch: 7,rmse: 0.016333332517938413
epoch: 8,rmse: 0.016320415775073295
epoch: 9,rmse: 0.016307516370429233
epoch: 10,rmse: 0.016294634277314257
epoch: 11,rmse: 0.01628176946912924
epoch: 12,rmse: 0.016268921919231256
epoch: 13,rmse: 0.016256091601075284
epoch: 14,rmse: 0.01624327848813133
epoch: 15,rmse: 0.016230482553920435
epoch: 16,rmse: 0.016217703772034804
epoch: 17,rmse: 0.016204942116039053
epoch: 18,rmse: 0.01619219755962522
epoch: 19,rmse: 0.016179470076432275
epoch: 20,rmse: 0.016166759640196385
epoch: 21,rmse: 0.01615406622471159
epoch: 22,rmse: 0.016141389803727552
epoch: 23,rmse: 0.016128730351158627
epoch: 24,rmse: 0.01611608784084588
epoch: 25,rmse: 0.016103462246732207
epoch: 26,rmse: 0.01609085354281742
epoch: 27,rmse: 0.0

# Gradient Descent in Pytorch

In [6]:
import torch

# 텐서로 변환
ratings_tensor=torch.from_numpy(ratings)

alpha=torch.tensor(ratings.mean(),requires_grad=True)
user_bias=torch.zeros(users.max()+1,requires_grad=True)
item_bias=torch.zeros(items.max()+1,requires_grad=True)

optim=torch.optim.SGD([alpha,user_bias,item_bias],lr=0.3) # SGD 사용 가능

lmd=0.001

for epoch in range(100):

    h=alpha + user_bias[users] + item_bias[items]
    mse=((h-ratings_tensor)**2).mean()
    reg=lmd*((item_bias**2).mean()+(user_bias**2).mean())
    cost=mse+reg

    optim.zero_grad()
    cost.backward()
    optim.step()

    with torch.no_grad():
        rmse=((h-ratings_tensor)**2).mean()**0.5
        print(f"epoch:{epoch} , rmse: {rmse}")

epoch:0 , rmse: 1.0607439399275533
epoch:1 , rmse: 1.0606711709580678
epoch:2 , rmse: 1.0605985323023008
epoch:3 , rmse: 1.0605260173436242
epoch:4 , rmse: 1.0604536199961365
epoch:5 , rmse: 1.0603813395699944
epoch:6 , rmse: 1.060309181345877
epoch:7 , rmse: 1.0602371369842036
epoch:8 , rmse: 1.0601652089809763
epoch:9 , rmse: 1.0600933992622092
epoch:10 , rmse: 1.060021707019839
epoch:11 , rmse: 1.0599501325977585
epoch:12 , rmse: 1.0598786738565378
epoch:13 , rmse: 1.059807328831113
epoch:14 , rmse: 1.059736105748738
epoch:15 , rmse: 1.0596649974842431
epoch:16 , rmse: 1.0595940003656408
epoch:17 , rmse: 1.059523121317959
epoch:18 , rmse: 1.059452357861111
epoch:19 , rmse: 1.0593817074011362
epoch:20 , rmse: 1.0593111699030173
epoch:21 , rmse: 1.0592407493219316
epoch:22 , rmse: 1.0591704444439194
epoch:23 , rmse: 1.0591002480214458
epoch:24 , rmse: 1.059030163687667
epoch:25 , rmse: 1.058960198831325
epoch:26 , rmse: 1.0588903458728602
epoch:27 , rmse: 1.058820601630013
epoch:28 , 

# Closed-form Solution
: 기울기가 0 이 되는 지점을 찾기

In [7]:
from tqdm import tqdm

lmd=0.001

alpha=ratings.mean()
user_bias=np.zeros(users.max()+1)
item_bias=np.zeros(items.max()+1)

for epoch in range(10):
    h=alpha+user_bias[users]+item_bias[items]
    rmse=((h-ratings)**2).mean()**0.5
    print(f"epoch:{epoch},rmse:{rmse}")

    alpha=(ratings-(user_bias[users]+item_bias[items])).mean()
    user_bias=np.bincount(users,weights=ratings-(alpha+item_bias[items]))/(np.bincount(users)+lmd)
    item_bias=np.bincount(items,weights=ratings-(alpha+user_bias[users]))/(np.bincount(items)+lmd)

h=alpha+user_bias[users]+item_bias[items]
rmse=((h-ratings)**2).mean()**0.5
print(f"final rmse:{rmse}")


epoch:0,rmse:1.0607439399275531
epoch:1,rmse:0.8663159834756426
epoch:2,rmse:0.8510867185839471
epoch:3,rmse:0.8503568292676987
epoch:4,rmse:0.8503078899843288
epoch:5,rmse:0.8503025377563744
epoch:6,rmse:0.8503016450633218
epoch:7,rmse:0.8503014646716495
epoch:8,rmse:0.8503014258804191
epoch:9,rmse:0.8503014173907169
final rmse:0.8503014155289849


gradient 보다 훨씬 빠르게 수렴한다